# Notebook 02 — Modèle TF-IDF

**Milestone M-02 | Étudiant B — IR Specialist**

Ce notebook couvre :
1. Indexation TF-IDF avec `TfidfVectorizer`
2. Fonction de recherche `search(query, k)` basée sur la similarité cosinus
3. Évaluation : Recall@10, Precision@10, MRR

## 1. Imports

In [1]:
import json
import os
import pickle

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR    = '../data'
OUTPUTS_DIR = '../outputs'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print('Imports OK')

Imports OK


## 2. Chargement des Données Préprocessées

In [2]:
# Chargement du DataFrame produit par 01_eda.ipynb
df_docs = pd.read_pickle(os.path.join(DATA_DIR, 'df_docs_preprocessed.pkl'))

with open(os.path.join(DATA_DIR, 'queries_train.json'), 'r', encoding='utf-8') as f:
    queries_train = json.load(f)

with open(os.path.join(DATA_DIR, 'qgts_train.json'), 'r', encoding='utf-8') as f:
    qgts_train = json.load(f)  # dict : query_id -> {total_relevant_docs, relevant_doc_ids}

with open(os.path.join(DATA_DIR, 'queries_test.json'), 'r', encoding='utf-8') as f:
    queries_test = json.load(f)

# Mapping id -> index numérique
id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df_docs['id'])}
idx_to_id = {idx: doc_id for doc_id, idx in id_to_idx.items()}

corpus = df_docs['content_clean'].tolist()

def get_relevant_ids(qgts, query_id):
    """Extrait les doc_ids pertinents depuis le ground truth."""
    if query_id not in qgts:
        return []
    return [e['doc_id'] for e in qgts[query_id]['relevant_doc_ids']]

def build_query_text(q):
    """Construit le texte de recherche depuis une entrée de requête."""
    parts = [q.get('text', ''), q.get('title', '')]
    if q.get('tags'):
        parts.append(' '.join(q['tags']))
    return ' '.join(p for p in parts if p).strip()

print(f'Corpus chargé          : {len(corpus):,} documents')
print(f'Requêtes train         : {len(queries_train)}')
print(f'Requêtes test          : {len(queries_test)}')

Corpus chargé          : 216,041 documents
Requêtes train         : 327
Requêtes test          : 141


## 3. Indexation TF-IDF

In [3]:
# Construction de l'index TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrammes + bigrammes
    min_df=1,
    max_df=0.95,
    sublinear_tf=True     # log(TF) — réduit l'impact des termes très fréquents
)

tfidf_matrix = vectorizer.fit_transform(corpus)  # (n_docs, n_features)

print(f'Matrice TF-IDF : {tfidf_matrix.shape}')
print(f'Nombre de features (termes) : {len(vectorizer.get_feature_names_out())}')

Matrice TF-IDF : (216041, 6495428)


Nombre de features (termes) : 6495428


## 4. Fonction de Recherche

In [4]:
def search(query: str, k: int = 10) -> dict:
    """
    Recherche TF-IDF par similarité cosinus.

    Args:
        query: Requête en langage naturel.
        k: Nombre de résultats à retourner.

    Returns:
        dict avec 'topk_indices' (indices dans df_docs) et 'topk_scores'.
    """
    query_vec = vectorizer.transform([query])
    scores    = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_k_idx = np.argsort(scores)[::-1][:k]
    return {
        'topk_indices': top_k_idx.tolist(),
        'topk_scores':  scores[top_k_idx].tolist()
    }

# Test rapide
result = search('neural networks deep learning', k=5)
print('Top 5 résultats pour "neural networks deep learning" :')
for idx, score in zip(result['topk_indices'], result['topk_scores']):
    print(f"  [{score:.4f}] {df_docs.iloc[idx]['id']} — {df_docs.iloc[idx]['title']}")

Top 5 résultats pour "neural networks deep learning" :
  [0.3652] d1656f91-cbd0-4985-9ba6-7ad90646b7cb_16249 — Are neural networks the best approach to artificial intelligence and machine learning?
  [0.3586] 31cb537a-5f48-4bcb-a246-2abac3e52023_72093 — What is a Neural Network in simple words
  [0.2063] e126a346-a719-4f2c-8e9e-035292f04abe_228985 — Could one sample be enough for a perceptron training?
  [0.2045] d5a7ce7d-2b90-4797-910b-5e8bb47b1f1c_222516 — Support Vector Machines as Neural Nets?
  [0.1903] 959d1675-7ab8-47c2-8ab9-d3af3b0ac6ee_65397 — Can symbolic AI 'learn' a data model?


## 5. Métriques d'Évaluation

In [5]:
def recall_at_k(retrieved_indices, relevant_ids, k, id_to_idx):
    """Recall@k : fraction des documents pertinents retrouvés dans le top-k."""
    relevant_set = set(id_to_idx[rid] for rid in relevant_ids if rid in id_to_idx)
    retrieved_set = set(retrieved_indices[:k])
    if not relevant_set:
        return 0.0
    return len(retrieved_set & relevant_set) / len(relevant_set)


def precision_at_k(retrieved_indices, relevant_ids, k, id_to_idx):
    """Precision@k : fraction du top-k qui est pertinente."""
    relevant_set = set(id_to_idx[rid] for rid in relevant_ids if rid in id_to_idx)
    retrieved_set = set(retrieved_indices[:k])
    return len(retrieved_set & relevant_set) / k


def mrr(retrieved_indices, relevant_ids, id_to_idx):
    """Mean Reciprocal Rank : 1/rang du premier document pertinent trouvé."""
    relevant_set = set(id_to_idx[rid] for rid in relevant_ids if rid in id_to_idx)
    for rank, idx in enumerate(retrieved_indices, start=1):
        if idx in relevant_set:
            return 1.0 / rank
    return 0.0

print('Fonctions de métriques définies : recall_at_k, precision_at_k, mrr')

Fonctions de métriques définies : recall_at_k, precision_at_k, mrr


## 6. Évaluation sur les Requêtes d'Entraînement

In [6]:
K = 10
recalls, precisions, mrrs_list = [], [], []

for query_entry in queries_train:
    query_id     = query_entry['id']
    query_text   = build_query_text(query_entry)
    relevant_ids = get_relevant_ids(qgts_train, query_id)

    if not relevant_ids:
        continue

    result    = search(query_text, k=K)
    retrieved = result['topk_indices']

    recalls.append(recall_at_k(retrieved, relevant_ids, K, id_to_idx))
    precisions.append(precision_at_k(retrieved, relevant_ids, K, id_to_idx))
    mrrs_list.append(mrr(retrieved, relevant_ids, id_to_idx))

tfidf_results = {
    'model':           'TF-IDF',
    f'Recall@{K}':     np.mean(recalls),
    f'Precision@{K}':  np.mean(precisions),
    'MRR':             np.mean(mrrs_list)
}

print(f'Évaluation sur {len(recalls)} requêtes (avec ground truth)')
print('=== Résultats TF-IDF ===')
for key, val in tfidf_results.items():
    if isinstance(val, float):
        print(f'{key:15s}: {val:.4f}')
    else:
        print(f'{key:15s}: {val}')

Évaluation sur 327 requêtes (avec ground truth)
=== Résultats TF-IDF ===
model          : TF-IDF
Recall@10      : 0.1052
Precision@10   : 0.0615
MRR            : 0.1514


In [7]:
# Sauvegarde des résultats pour le notebook de consolidation
with open(os.path.join(OUTPUTS_DIR, 'tfidf_results.pkl'), 'wb') as f:
    pickle.dump(tfidf_results, f)

# Sauvegarde du vectorizer pour réutilisation (classifieur, etc.)
with open(os.path.join('../models', 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(vectorizer, f)
with open(os.path.join('../models', 'tfidf_matrix.pkl'), 'wb') as f:
    pickle.dump(tfidf_matrix, f)

print('Résultats et modèle sauvegardés.')

Résultats et modèle sauvegardés.


## 7. Analyse Qualitative

In [8]:
# Analyse sur 3 requêtes d'entraînement
sample_queries = queries_train[:3]

for q_entry in sample_queries:
    q_text   = build_query_text(q_entry)
    q_id     = q_entry['id']
    relevant = get_relevant_ids(qgts_train, q_id)
    result   = search(q_text, k=5)

    print(f'Requête : "{q_text[:80]}"')
    print(f'Docs pertinents ({len(relevant)}) : {relevant[:3]}...')
    print('Top-5 TF-IDF :')
    for idx, score in zip(result['topk_indices'], result['topk_scores']):
        doc_id = idx_to_id[idx]
        marker = ' ✓' if doc_id in relevant else ''
        print(f"  [{score:.4f}] {doc_id} — {df_docs.iloc[idx]['title'][:60]}{marker}")
    print()

Requête : "Want to try reformatting Damaged SD Card linux development"
Docs pertinents (4) : ['135f5fdb-bcba-40b8-b90d-823617f1e805_21141', 'd0609dab-fbf2-4838-be6a-05e76b291258_30322', 'cccd8617-6417-44a5-9f67-d4b7001b7165_47263']...
Top-5 TF-IDF :
  [0.1716] 569b7bce-60ab-4ec6-9baa-33dfd9a5489f_87449 — Linux development server
  [0.1685] 48ce31f4-a9eb-49df-9e8f-da696b70d124_166481 — Difference between Windows and Linux development environment
  [0.1477] 18c08039-f948-4702-b217-3a527c39a03f_77319 — Starting Linux Programming
  [0.1341] 276bf1e2-20da-4758-9009-d2472ad21266_17022 — "USB Storage damaged. It may need reformatting" error messag
  [0.1327] e4994c74-aaf7-400a-b417-cf344c16010f_120465 — How to determine how many bytes / hr are being written to my



Requête : "Convince grep to output all lines, not just those with matches shell virtualizat"
Docs pertinents (5) : ['ada9aadc-7371-450f-b610-9c1b40c8bedc_106565', '224d514e-67e9-4d63-b24e-a0d7b2d205ee_146077', '744e7467-dfbb-4e7a-90ad-4e56998114d2_103281']...
Top-5 TF-IDF :
  [0.1217] 198f22ba-a491-4fa9-bcc0-4dbca8c3e380_152484 — Loop in WildCard as Input of Script
  [0.1178] 2ac9a961-db1c-4367-a90c-d355a1498cf5_38346 — Getting from proficient to expert
  [0.0882] fb4558fa-e0db-4dc6-8bef-28905c358520_55298 — Output multiple files from a single grep?
  [0.0880] aa6b06c1-b274-46a3-a10d-530cc7443568_80996 — How can I mount vxfs FS to two or more Solaris servers?
  [0.0870] 7126a671-70e4-4a06-8246-0435f92bca4b_57876 — grep the only first word from output with grep -P



Requête : "How can I automatically reject some types of calls? linux development"
Docs pertinents (4) : ['ed189b8f-e6fe-42e3-b326-333b86d979b5_9977', '4df54059-3169-44c8-b650-5aab3e0115ca_48997', '6b89b8a1-5f98-420a-8347-5efea3e4740c_15533']...
Top-5 TF-IDF :
  [0.1460] 48ce31f4-a9eb-49df-9e8f-da696b70d124_166481 — Difference between Windows and Linux development environment
  [0.1446] 569b7bce-60ab-4ec6-9baa-33dfd9a5489f_87449 — Linux development server
  [0.1331] 4d31b78c-2939-45da-8567-9789f0392193_12275 — How To Stop Rejecting Messages At Low Memory?
  [0.1224] 18c08039-f948-4702-b217-3a527c39a03f_77319 — Starting Linux Programming
  [0.1185] 34cf059f-742c-4dd5-86c1-ed8d9e28717d_11702 — How to record Skype video calls on Linux?



## 8. Résumé

Le modèle TF-IDF :
- Utilise `TfidfVectorizer` avec unigrammes et bigrammes, `sublinear_tf=True`
- Recherche par similarité cosinus entre la requête et la matrice de documents
- Interface standardisée : `search(query, k) -> {'topk_indices': [...], 'topk_scores': [...]}`

Les résultats sont sauvegardés dans `outputs/tfidf_results.pkl`.